## Bronze layer

In [0]:
df = spark.read.csv('/Volumes/development/lake/lake_practical/raw_orders.csv', header=True)
display(df)

In [0]:
from pyspark.sql.functions import current_date
df_bronze = df.withColumn("ingestion_timestamp", current_date())
display(df_bronze)

## Delta Table

In [0]:
# Write to silver Delta Table
df_bronze.write.format("delta").mode("overwrite").saveAsTable("development.lake.bronze_orders")

## Silver layer & Delta Table

In [0]:
from pyspark.sql.functions import col, upper
df_silver = (df_bronze
.dropDuplicates(subset=['order_id'])
.dropna(subset=["amount"])
.withColumn('status', upper(col('status')))
)

(df_silver
.write
.mode("overwrite")
.format("delta")
.saveAsTable("development.lake.silver_orders"))